In [1]:
import logging
import torch
import numpy as np
import torch.nn.functional as F

from torch import nn
from torchvision import datasets, transforms

In [2]:
log = logging.getLogger('cnn')
log.setLevel(logging.INFO)

formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
stream_handler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
log.addHandler(stream_handler)

For resnet implemention, I have referenced following github repository<br>
https://github.com/pytorch/vision/blob/master/torchvision/models/resnet.py

In [3]:
class Bottleneck(nn.Module):
    expansion = 4
    def __init__(self, in_channel, n_filters, downsample=None, stride=1):
        super(Bottleneck, self).__init__()
        # first 1x1
        self.cv1 = nn.Conv2d(in_channel, n_filters, kernel_size=1)
        self.bn1 = nn.BatchNorm2d(n_filters)

        # second 3x3
        # FIXME: padding=1 and stride=2 should be controlled
        self.cv2 = nn.Conv2d(n_filters, n_filters, kernel_size=3, padding=1, stride=stride)
        self.bn2 = nn.BatchNorm2d(n_filters)

        # third 1x1
        self.cv3 = nn.Conv2d(n_filters, n_filters*self.expansion, kernel_size=1)
        self.bn3 = nn.BatchNorm2d(n_filters*self.expansion)
        self.relu = nn.ReLU()

        # downsample
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = self.cv1(x)
        out = self.bn1(out)
        log.debug('cv1 result: {}'.format(out.shape))

        out = self.cv2(out)
        out = self.bn2(out)
        log.debug('cv2 result: {}'.format(out.shape))

        out = self.cv3(out)
        out = self.bn3(out)
        out = self.relu(out)
        log.debug('cv3 result: {}'.format(out.shape))

        # do something about downsample here
        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        log.debug('out: {}'.format(out.shape))
        log.debug('identity: {}'.format(identity.shape))
        return out

In [4]:
def get_padding(i):
    return int(np.ceil(i/2) - 1)

In [5]:
class Resnet(nn.Module):
    def __init__(self, bottleneck, layers, num_classes=1000, zero_init_residual=False, in_channel=3):
        super(Resnet, self).__init__()

        self.planes = 64
        k = 7
        n_pad = get_padding(k)
        self.cv1 = nn.Conv2d(in_channel, self.planes, kernel_size=k, padding=n_pad, stride=2, bias=False)
        self.bn1 = nn.BatchNorm2d(self.planes)
        self.relu = nn.ReLU()
        self.mx1 = nn.MaxPool2d(kernel_size=3, stride=2, padding=get_padding(3))

        # add bottleneck layer
        self.layer1 = self._add_bottleneck_layer(bottleneck, 64, layers[0])
        self.layer2 = self._add_bottleneck_layer(bottleneck, 128, layers[1], stride=2)
        self.layer3 = self._add_bottleneck_layer(bottleneck, 256, layers[2], stride=2)
        self.layer4 = self._add_bottleneck_layer(bottleneck, 512, layers[3], stride=2)

        # average pooling and fully-connected layer
        self.avgpool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(512*bottleneck.expansion, num_classes)

    def _add_bottleneck_layer(self, block, out_planes, n_block, stride=1):
        downsample = None
        if stride != 1 or self.planes != out_planes*block.expansion:
            # do downsampling
            downsample = nn.Sequential(*[
                nn.Conv2d(self.planes, out_planes*block.expansion, kernel_size=1, stride=stride),
                nn.BatchNorm2d(out_planes*block.expansion)
            ])

        layers = []
        layers.append(block(self.planes, out_planes, downsample=downsample, stride=stride))
        self.planes = out_planes*block.expansion
        for _ in range(1, n_block):
            layers.append(block(self.planes, out_planes))
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.cv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.mx1(out)
        
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        
        # for classifier
        out = self.avgpool(out)
        out = self.fc(out.reshape(out.shape[0], -1))
        
        # FIXME: do something more here
        return out

define model

In [6]:
bottelneck = Bottleneck
n_class = 10
model = Resnet(bottelneck, [3,4,5,3], 10, in_channel=1)

In [7]:
# parameter definition
batch_size = 128
log_interval = 100

make train/test DataLoader

In [8]:
train_loader = torch.utils.data.DataLoader(
        datasets.MNIST('/data/mnist', train=True, download=True,
                       transform=transforms.Compose([
                           transforms.ToTensor(),
                           transforms.Normalize((0.1307,), (0.3081,))
                       ])),
        batch_size=batch_size, shuffle=True)

In [9]:
test_loader = torch.utils.data.DataLoader(
        datasets.MNIST('/data/mnist', train=False, transform=transforms.Compose([
                           transforms.ToTensor(),
                           transforms.Normalize((0.1307,), (0.3081,))
                       ])),
        batch_size=1000, shuffle=True)

set GPU device

In [10]:
device = torch.device('cuda')
model = model.to(device)

In [11]:
import torch.optim as opt
optimizer = opt.SGD(model.parameters(), lr=0.01, momentum=0.5)
optimizer

SGD (
Parameter Group 0
    dampening: 0
    lr: 0.01
    momentum: 0.5
    nesterov: False
    weight_decay: 0
)

define train/test functiton

In [12]:
def train(model, device, train_loader, optimizer, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss_func = nn.CrossEntropyLoss(ignore_index=-1)
        loss = loss_func(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % log_interval == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))

def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.nll_loss(output, target, reduction='sum').item() # sum up batch loss
            pred = output.argmax(dim=1, keepdim=True) # get the index of the max log-probability
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)

    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        test_loss, correct, len(test_loader.dataset),
        100. * correct / len(test_loader.dataset)))

In [13]:
epochs = 3
for epoch in range(1, epochs+1):
    train(model, device, train_loader, optimizer, epoch)
    test(model, device, test_loader)

Train Epoch: 1 [0/60000 (0%)]	Loss: 2.374273
Train Epoch: 1 [12800/60000 (21%)]	Loss: 0.183302
Train Epoch: 1 [25600/60000 (43%)]	Loss: 0.140149
Train Epoch: 1 [38400/60000 (64%)]	Loss: 0.083886
Train Epoch: 1 [51200/60000 (85%)]	Loss: 0.054406

Test set: Average loss: -9.7556, Accuracy: 9859/10000 (99%)

Train Epoch: 2 [0/60000 (0%)]	Loss: 0.039430
Train Epoch: 2 [12800/60000 (21%)]	Loss: 0.033964
Train Epoch: 2 [25600/60000 (43%)]	Loss: 0.017713
Train Epoch: 2 [38400/60000 (64%)]	Loss: 0.039662
Train Epoch: 2 [51200/60000 (85%)]	Loss: 0.017541

Test set: Average loss: -11.3800, Accuracy: 9890/10000 (99%)

Train Epoch: 3 [0/60000 (0%)]	Loss: 0.017404
Train Epoch: 3 [12800/60000 (21%)]	Loss: 0.005796
Train Epoch: 3 [25600/60000 (43%)]	Loss: 0.014076
Train Epoch: 3 [38400/60000 (64%)]	Loss: 0.003913
Train Epoch: 3 [51200/60000 (85%)]	Loss: 0.006572

Test set: Average loss: -12.0812, Accuracy: 9905/10000 (99%)

